In [1]:
import requests
import pandas as pd

OTP_URL = "http://localhost:8080/otp/gtfs/v1"

def query_otp(query: str, variables: dict | None = None) -> dict:
    """Run a GraphQL query against the OTP server and return the `data` payload."""
    resp = requests.post(
        OTP_URL,
        json={"query": query, "variables": variables or {}},
        headers={"Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    payload = resp.json()
    if "errors" in payload:
        raise RuntimeError(payload["errors"])
    return payload["data"]

In [2]:
PLAN_QUERY = """
query Plan(
  $from: InputCoordinates!
  $to: InputCoordinates!
  $date: String!
  $time: String!
  $modes: [TransportMode]
  $numItineraries: Int
  $searchWindow: Long
) {
  plan(
    from: $from
    to: $to
    date: $date
    time: $time
    transportModes: $modes
    numItineraries: $numItineraries
    searchWindow: $searchWindow
  ) {
    itineraries {
      start
      end
      duration
      numberOfTransfers
      legs {
        mode
        distance
        duration
        start { scheduledTime }
        end { scheduledTime }
        route { shortName longName }
        from { name }
        to { name }
        fareProducts {
          product {
            name
            ... on DefaultFareProduct {
              price { amount currency { code } }
            }
          }
        }
      }
    }
  }
}
"""

REFERENCE_DATE = "2026-05-20"  # Wednesday 20 May 2026 -- project reference day
DEPART_TIME = "08:00am"

MELBOURNE_CBD = {"lat": -37.8183, "lon": 144.9671}    # Southern Cross Station area
SYDNEY_CBD = {"lat": -33.8688, "lon": 151.2093}       # Central Station area
MELBOURNE_AIRPORT = {"lat": -37.669, "lon": 144.8478}
SYDNEY_AIRPORT = {"lat": -33.93901, "lon": 151.1740}
SOUTHERN_CROSS_STATION = {"lat": -37.8183, "lon": 144.9526}  # Southern Cross Station (Melbourne)
CENTRAL_STATION_SYDNEY = {"lat": -33.8830, "lon": 151.2070}  # Central Station (Sydney)
AVALON_AIRPORT = {"lat": -38.0394, "lon": 144.4695}




In [3]:
from datetime import timedelta


def format_duration(seconds: float | None) -> str | None:
    """e.g. 5025 -> '1h 23m', 180 -> '3m'"""
    if seconds is None:
        return None
    total_min = round(seconds / 60)
    h, m = divmod(total_min, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m"


def leg_fare(leg: dict) -> tuple[float | None, str | None]:
    for fp in leg["fareProducts"]:
        price = fp["product"].get("price")
        if price:
            return price["amount"], price["currency"]["code"]
    return None, None


def itineraries_to_df(itineraries: list[dict]) -> pd.DataFrame:
    """Aggregate a list of `plan` itineraries into one summary row each.

    Stamps each itinerary dict with a stable `id` (if it doesn't already have
    one) so it can later be looked up with `show_itinerary`, independent of
    however the resulting DataFrame gets sorted or filtered.
    """
    rows = []
    for i, itin in enumerate(itineraries):
        itin.setdefault("id", i)
        legs = itin["legs"]
        transit_legs = [leg for leg in legs if leg["route"] is not None]
        modes_used = list(dict.fromkeys(leg["mode"] for leg in transit_legs))

        total_fare, currency = 0.0, None
        for leg in legs:
            amount, cur = leg_fare(leg)
            if amount:
                total_fare += amount
                currency = currency or cur

        rows.append({
            "id": itin["id"],
            **({"requested_mode": itin["requested_mode"]} if "requested_mode" in itin else {}),
            "origin": legs[0]["from"]["name"],
            "destination": legs[-1]["to"]["name"],
            "depart": itin["start"],
            "arrive": itin["end"],
            "duration": format_duration(itin["duration"]),
            "transfers": itin["numberOfTransfers"],
            "modes": " -> ".join(modes_used) if modes_used else "WALK",
            "distance_km": round(sum(leg["distance"] for leg in legs if leg["distance"]) / 1000, 2),
            "total_fare": round(total_fare, 2) if currency else None,
            "currency": currency,
        })

    df = pd.DataFrame(rows)
    df["depart"] = pd.to_datetime(df["depart"]).dt.strftime("%Y-%m-%d %H:%M")
    df["arrive"] = pd.to_datetime(df["arrive"]).dt.strftime("%Y-%m-%d %H:%M")
    return df


def show_itinerary(itineraries: list[dict], itinerary_id: int) -> None:
    """Print full leg-by-leg detail for one itinerary, looked up by its `id`
    (the same id shown in the `itineraries_to_df` output)."""
    itin = next((it for it in itineraries if it.get("id") == itinerary_id), None)
    if itin is None:
        raise ValueError(f"No itinerary with id={itinerary_id}")

    total_fare, currency = 0.0, None
    for leg in itin["legs"]:
        amount, cur = leg_fare(leg)
        if amount:
            total_fare += amount
            currency = currency or cur

    header = f"Itinerary {itinerary_id}"
    if "requested_mode" in itin:
        header += f" ({itin['requested_mode']})"
    print(header)
    print(f"  {itin['start']}  ->  {itin['end']}  "
          f"({format_duration(itin['duration'])}, {itin['numberOfTransfers']} transfer(s))")
    if currency:
        print(f"  Total fare: {total_fare:.2f} {currency}")
    print()

    for i, leg in enumerate(itin["legs"], 1):
        route = leg["route"]["shortName"] if leg["route"] else None
        fare_amount, fare_currency = leg_fare(leg)
        fare_str = f"{fare_amount:.2f} {fare_currency}" if fare_amount else "-"
        distance_km = round(leg["distance"] / 1000, 2) if leg["distance"] else None

        label = leg["mode"] + (f" [{route}]" if route else "")
        print(f"  Leg {i}: {label}")
        print(f"    {leg['from']['name']} ({leg['start']['scheduledTime']})"
              f"  ->  {leg['to']['name']} ({leg['end']['scheduledTime']})")
        detail = f"    duration: {format_duration(leg['duration'])}"
        if distance_km:
            detail += f", distance: {distance_km} km"
        detail += f", fare: {fare_str}"
        print(detail)


def get_all_itineraries_for_day(
    from_coords: dict,
    to_coords: dict,
    modes: list[dict],
    date: str,
    day_start: str = "12:00am",
    itineraries_per_call: int = 5,
    empty_probe_step: timedelta = timedelta(hours=1),
    max_calls: int = 60,
) -> list[dict]:
    """Page OTP's `plan` query across a full day to collect every distinct itinerary.

    OTP's `plan` search has a limited look-ahead window, so an empty result at
    a given time doesn't mean the day's service is exhausted -- it can just
    mean the next departure is further away than the window reaches (e.g. an
    overnight gap before the first morning service). On an empty result we
    skip ahead by `empty_probe_step` and retry rather than stopping.
    """
    search_time = pd.Timestamp(f"{date} {day_start}")
    day_end = search_time + timedelta(days=1)

    seen: set[tuple] = set()
    all_itineraries: list[dict] = []

    for _ in range(max_calls):
        if search_time >= day_end:
            break
        variables = {
            "from": from_coords,
            "to": to_coords,
            "date": search_time.strftime("%Y-%m-%d"),
            "time": search_time.strftime("%I:%M%p").lower(),
            "modes": modes,
            "numItineraries": itineraries_per_call,
        }
        found = query_otp(PLAN_QUERY, variables)["plan"]["itineraries"]
        if not found:
            search_time += empty_probe_step
            continue

        departures = []
        for itin in found:
            key = tuple(
                (leg["mode"], leg["route"]["shortName"] if leg["route"] else None,
                 leg["start"]["scheduledTime"], leg["end"]["scheduledTime"])
                for leg in itin["legs"]
            )
            depart = pd.Timestamp(itin["start"]).tz_localize(None)  # OTP returns tz-aware local time; keep wall clock, drop offset
            departures.append(depart)
            if key not in seen and depart < day_end:
                seen.add(key)
                all_itineraries.append(itin)

        # Always advance past the earliest departure in this batch (by at least
        # a minute) so a batch of all-duplicates can't loop forever.
        search_time = max(min(departures), search_time) + timedelta(minutes=1)

    return all_itineraries

In [4]:
# Example: compare CAR / RAIL / AIR at a single fixed departure time.
# Each mode can override numItineraries and/or searchWindow (seconds) --
# e.g. AIR below returns every flight departing in the next 3 hours.
MODE_QUERIES = {
    "CAR": {
        "from": MELBOURNE_CBD,
        "to": SYDNEY_CBD,
        "modes": [{"mode": "CAR"}],
        "numItineraries": 3,
    },
    "RAIL": {
        "from": SOUTHERN_CROSS_STATION,
        "to": CENTRAL_STATION_SYDNEY,
        "modes": [{"mode": "WALK"}, {"mode": "RAIL"}],
        "numItineraries": 1,
        "searchWindow" : 24 * 60 * 60
    },
    "AIR": {
        "from": MELBOURNE_AIRPORT,
        "to": SYDNEY_AIRPORT,
        "modes": [{"mode": "WALK"}, {"mode": "AIRPLANE"}],
        "numItineraries": 2,       # high enough that searchWindow -- not this cap -- limits results
        "searchWindow": 3 * 60 * 60,  # 3 hours
    },
    "COACH": {
        "from": SOUTHERN_CROSS_STATION,
        "to": CENTRAL_STATION_SYDNEY,
        "modes": [{"mode": "WALK"}, {"mode": "BUS"}],
        "numItineraries": 10,
        "searchWindow": 24 * 60 * 60,  # 12 hours to capture all feasible intercity coaches
    },
}

itineraries = []
for requested_mode, cfg in MODE_QUERIES.items():
    variables = {
        "from": cfg["from"],
        "to": cfg["to"],
        "date": REFERENCE_DATE,
        "time": DEPART_TIME,
        "modes": cfg["modes"],
        "numItineraries": cfg.get("numItineraries", 3),
    }
    if "searchWindow" in cfg:
        variables["searchWindow"] = cfg["searchWindow"]
    found = query_otp(PLAN_QUERY, variables)["plan"]["itineraries"]
    for itin in found:
        itin["requested_mode"] = requested_mode
        itineraries.append(itin)

routes_df = itineraries_to_df(itineraries)
routes_df

,id,requested_mode,origin,destination,depart,arrive,duration,transfers,modes,distance_km,total_fare,currency
0,0,RAIL,Origin,Destination,2026-05-20 08:26,2026-05-20 19:51,11h 24m,0,RAIL,957.19,110.0,AUD
1,1,AIR,Melbourne Airport (Tullamarine),Sydney Kingsford Smith Airport,2026-05-20 08:00,2026-05-20 09:30,1h 30m,0,AIRPLANE,705.05,170.0,AUD
2,2,AIR,Melbourne Airport (Tullamarine),Sydney Kingsford Smith Airport,2026-05-20 08:15,2026-05-20 09:45,1h 30m,0,AIRPLANE,705.05,110.0,AUD
3,3,COACH,Origin,Destination,2026-05-20 18:54,2026-05-21 06:12,11h 18m,0,BUS,766.31,80.0,AUD
4,4,COACH,Origin,Destination,2026-05-20 21:54,2026-05-21 10:02,12h 08m,0,BUS,725.36,98.0,AUD


In [6]:
show_itinerary(itineraries, itinerary_id=3)

Itinerary 3 (COACH)
  2026-05-20T18:54:23+10:00  ->  2026-05-21T06:12:05+10:00  (11h 18m, 0 transfer(s))
  Total fare: 80.00 AUD

  Leg 1: WALK
    Origin (2026-05-20T18:54:23+10:00)  ->  Melbourne Southern Cross Coach Terminal (2026-05-20T19:00:00+10:00)
    duration: 6m, distance: 0.21 km, fare: -
  Leg 2: BUS [FLY]
    Melbourne Southern Cross Coach Terminal (2026-05-20T19:00:00+10:00)  ->  Sydney Central (Pitt St Coach) (2026-05-21T06:10:00+10:00)
    duration: 11h 10m, distance: 765.99 km, fare: 80.00 AUD
  Leg 3: WALK
    Sydney Central (Pitt St Coach) (2026-05-21T06:10:00+10:00)  ->  Destination (2026-05-21T06:12:05+10:00)
    duration: 2m, distance: 0.12 km, fare: -


### All itineraries for one mode across a full day

`plan` only returns the best options departing at or after a given `time` — it
doesn't enumerate every scheduled service. `get_all_itineraries_for_day`
(defined above) pages the search across the day to collect every distinct
itinerary for a mode.

In [7]:
# Example: every itinerary for a mode between Melbourne and Sydney on the reference day, showing the service provider for each result

# Fix for BUS mode: correctly handle bus mode naming (BUS vs. COACH) and ensure service provider extraction works for intercity bus results

MODE_TO_EXTRACT = "BUS"  # can be "AIRPLANE", "RAIL", "BUS", etc.
FROM = SOUTHERN_CROSS_STATION
TO = CENTRAL_STATION_SYDNEY
MODES = [{"mode": "WALK"}, {"mode": "BUS"}]  # adjust as needed for the mode
PROVIDER_COL_NAME = "service_provider"  # generic column name

# Collect all itineraries for the selected mode between FROM and TO on the reference day
day_itineraries = get_all_itineraries_for_day(
    from_coords=FROM,
    to_coords=TO,
    modes=MODES,
    date=REFERENCE_DATE,
)

day_df = itineraries_to_df(day_itineraries).sort_values("depart").reset_index(drop=True)
print(f"{MODE_TO_EXTRACT}: {len(day_itineraries)} distinct itineraries found across {REFERENCE_DATE}")

# Generic: extract service/provider for BUS mode (accounting for possible different keys)
def get_provider_for_row(row, mode=MODE_TO_EXTRACT, itineraries=day_itineraries):
    itinerary_id = row['id']
    itinerary = itineraries[int(itinerary_id)]
    for leg in itinerary.get('legs', []):
        # Handle various modes / names (OTP may return 'BUS' for urban, 'COACH' or 'BUS' for intercity)
        leg_mode = str(leg.get("mode", "")).upper()

        # Accept a match if either is bus or coach, to be robust to feed variations
        if leg_mode in ("BUS", "COACH") and mode in ("BUS", "COACH"):
            # Service provider may be under multiple keys depending on feed
            for key in ["operator", "agency", "airline"]:
                if key in leg and leg[key]:
                    return leg[key]
            # For coach/ferry/bus—sometimes shortName is the provider (Firefly, Greyhound, etc.)
            if "route" in leg and leg["route"]:
                if "shortName" in leg["route"] and leg["route"]["shortName"]:
                    return leg["route"]["shortName"]
                # Occasionally 'agency' or 'longName' is here
                if "agency" in leg["route"] and leg["route"]["agency"]:
                    return leg["route"]["agency"]
                if "longName" in leg["route"] and leg["route"]["longName"]:
                    return leg["route"]["longName"]
    return None

day_df[PROVIDER_COL_NAME] = day_df.apply(get_provider_for_row, axis=1)

day_df  # Displays with the service provider shown for the selected mode

# Compute statistics as before, but handle BUS/COACH and missing data
import numpy as np

def parse_duration_to_minutes(duration_str):
    """Helper to parse duration like '1h 24m' to minutes as float."""
    if not isinstance(duration_str, str):
        return np.nan
    h, m = 0, 0
    parts = duration_str.strip().split()
    for part in parts:
        if part.endswith('h'):
            try:
                h = int(part[:-1])
            except ValueError:
                h = 0
        elif part.endswith('m'):
            try:
                m = int(part[:-1])
            except ValueError:
                m = 0
    return h * 60 + m

total_daily_frequency = len(day_df)
avg_transfers = day_df['transfers'].astype(float).mean()
avg_fare = day_df['total_fare'].astype(float).mean()
avg_distance = day_df['distance_km'].astype(float).mean()
avg_duration_minutes = day_df['duration'].apply(parse_duration_to_minutes).mean()

print(f"{MODE_TO_EXTRACT} itinerary stats for", REFERENCE_DATE)
print(f"  Total itineraries (frequency): {total_daily_frequency}")
print(f"  Average number of transfers:   {avg_transfers:.2f}")
currency = day_df['currency'].iloc[0] if 'currency' in day_df and len(day_df['currency']) > 0 else ""
print(f"  Average fare:                 {avg_fare:.2f} {currency}")
print(f"  Average duration:             {avg_duration_minutes/60:.2f} h")
print(f"  Average distance:             {avg_distance:.2f} km")
print(f"  Providers present:", sorted(set(x for x in day_df[PROVIDER_COL_NAME] if x)))

BUS: 2 distinct itineraries found across 2026-05-20
BUS itinerary stats for 2026-05-20
  Total itineraries (frequency): 2
  Average number of transfers:   0.00
  Average fare:                 89.00 AUD
  Average duration:             11.72 h
  Average distance:             745.84 km
  Providers present: ['FLY', 'GX']
